In [44]:
import pandas as pd
import numpy as np
from collections import namedtuple
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [45]:
import inspect
import pandas as pd
import numpy as np
from collections import namedtuple
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ── Synthetic waveform generator ───────────────────────────────────────────────
def _adapt_rpm_profile_fn(profile):
    """Normalize profile to callable(angle_deg, elapsed_ms) -> float RPM."""
    if not callable(profile):
        rpm_value = float(profile)
        return lambda angle_deg, elapsed_ms: rpm_value

    try:
        sig = inspect.signature(profile)
        params = list(sig.parameters.values())
        has_varargs = any(p.kind == inspect.Parameter.VAR_POSITIONAL for p in params)
        positional_count = sum(
            p.kind in (inspect.Parameter.POSITIONAL_ONLY, inspect.Parameter.POSITIONAL_OR_KEYWORD)
            for p in params
        )
        accepts_time = has_varargs or positional_count >= 2
    except (TypeError, ValueError):
        accepts_time = False

    if accepts_time:
        return lambda angle_deg, elapsed_ms: float(profile(angle_deg, elapsed_ms))

    return lambda angle_deg, elapsed_ms: float(profile(angle_deg))


def gen_synthetic_waveform(rpm, n_revs, wheel, noise_ms=0.0):
    """
    Generate synthetic rising-edge timestamps for a trigger wheel.

    Parameters
    ----------
    rpm      : float or callable(cumulative_angle_deg[, elapsed_time_ms]) -> float
               Constant RPM (float) or a function returning instantaneous RPM
               as a function of cumulative crank angle [degrees], optionally
               with elapsed time [ms] as a second argument.
               Use the rpm_* helpers below to build common profiles.
    n_revs   : int            — number of full revolutions to simulate
    wheel    : WheelDescriptor — geometry of the trigger wheel
    noise_ms : float          — std-dev of Gaussian timing jitter added per edge [ms]

    Returns
    -------
    times     : pd.Series  — rising-edge timestamps [ms]
    intervals : pd.Series  — per-tooth durations     [ms]
    """
    rpm_fn = _adapt_rpm_profile_fn(rpm)
    tooth_angles = np.asarray(wheel.tooth_angles_deg, dtype=float)
    n_teeth = len(tooth_angles)
    n_steps = int(n_revs) * n_teeth

    times = np.empty(n_steps, dtype=float)
    intervals = np.empty(n_steps, dtype=float)

    idx = 0
    t_ms = 0.0
    angle_deg = 0.0

    for i in range(n_steps):
        times[i] = t_ms

        tooth_angle = tooth_angles[idx]
        rpm_now = rpm_fn(angle_deg, t_ms)
        if rpm_now <= 0.0:
            raise ValueError(
                f"RPM profile produced non-positive RPM={rpm_now:.6f} at angle={angle_deg:.3f}°, t={t_ms:.3f} ms"
            )

        dt_ms = tooth_angle * 60_000.0 / (360.0 * rpm_now)
        if noise_ms > 0.0:
            dt_ms = max(dt_ms + np.random.normal(0.0, noise_ms), 1e-6)

        intervals[i] = dt_ms
        t_ms += dt_ms
        angle_deg += tooth_angle
        idx = (idx + 1) % n_teeth

    return pd.Series(times), pd.Series(intervals)


def gen_synthetic_signal(times, intervals, duty=0.4):
    """
    Build a plottable square-wave signal from rising-edge timestamps and intervals.

    Parameters
    ----------
    times     : rising-edge timestamps [ms]
    intervals : per-tooth durations [ms]  (from gen_synthetic_waveform)
    duty      : fraction of each interval spent high (default 0.4)

    Returns
    -------
    (signal_times, signal_values) as pd.Series pairs
    """
    t = np.asarray(times, dtype=float)
    dt = np.asarray(intervals, dtype=float)
    n = len(t)

    sig_t = np.empty(3 * n, dtype=float)
    sig_v = np.empty(3 * n, dtype=float)

    sig_t[0::3] = t
    sig_t[1::3] = t
    sig_t[2::3] = t + dt * duty

    sig_v[0::3] = 0.0
    sig_v[1::3] = 1.0
    sig_v[2::3] = 0.0

    return pd.Series(sig_t), pd.Series(sig_v)


# ── RPM profile helpers ────────────────────────────────────────────────────────
def rpm_const(rpm):
    """Constant-speed profile."""
    return lambda angle: rpm


def rpm_linear(rpm_start, accel_rpm_per_rev):
    """
    Linearly-accelerating profile.
    accel_rpm_per_rev: RPM change per full revolution (positive = accelerating).
    """
    return lambda angle: rpm_start + accel_rpm_per_rev * (angle / 360.0)


def rpm_linear_ms(rpm_start, accel_rpm_per_ms):
    """Linearly-accelerating profile in elapsed time (ms)."""
    return lambda angle, t_ms: rpm_start + accel_rpm_per_ms * t_ms


def rpm_sine(rpm_base, amplitude, period_deg=360.0, phase_deg=0.0):
    """
    Sinusoidally-modulated RPM profile.
    rpm_base   : mean RPM
    amplitude  : peak RPM deviation (must be < rpm_base to keep RPM positive)
    period_deg : period of the sine modulation in crank-angle degrees
                 (default 360 = one oscillation per revolution)
    phase_deg  : phase offset [degrees]
    """
    phase_rad = np.deg2rad(phase_deg)
    return lambda angle: rpm_base + amplitude * np.sin(2 * np.pi * angle / period_deg + phase_rad)


def rpm_staged_ms(stages, fallback=None):
    """
    Compose a staged RPM profile over elapsed time.

    Parameters
    ----------
    stages : iterable of (duration_ms, profile_fn)
             profile_fn can be either callable(angle) or callable(angle, stage_time_ms).
             duration_ms can be None/np.inf for an open-ended stage.
    fallback : callable(angle[, stage_time_ms]) or None
               Used after all finite stages. If None, reuses the final stage profile.

    Returns
    -------
    callable(angle, time_ms) -> rpm
    """
    normalized = []
    t_start = 0.0

    for duration_ms, profile_fn in stages:
        duration = np.inf if duration_ms is None else float(duration_ms)
        if duration <= 0.0 and not np.isinf(duration):
            raise ValueError(f"Stage duration must be > 0 ms (or None/inf), got {duration}")

        t_end = np.inf if np.isinf(duration) else (t_start + duration)
        normalized.append((t_start, t_end, _adapt_rpm_profile_fn(profile_fn)))

        if np.isinf(duration):
            break
        t_start = t_end

    if not normalized:
        raise ValueError("At least one stage is required")

    fallback_fn = _adapt_rpm_profile_fn(fallback if fallback is not None else stages[-1][1])

    def profile(angle, time_ms):
        t = float(time_ms)
        for t0, t1, fn in normalized:
            if t < t1:
                return fn(angle, t - t0)

        last_end = normalized[-1][1]
        fallback_t = t - last_end if np.isfinite(last_end) else 0.0
        return fallback_fn(angle, fallback_t)

    return profile




In [67]:
# ── Synthetic data parameters ──────────────────────────────────────────────────
SYN_RPM          = 300.0   # base RPM
SYN_N_REVS       = 100      # number of full revolutions to simulate
SYN_TOOTH_OFFSET = 0       # tooth index assigned to the first edge by the filter

# ── Wheel geometry ─────────────────────────────────────────────────────────────
# Explicit crank angle [deg] from each real tooth to the next.
# Edit individual values here to model non-uniform tooth spacing.
# Must sum to 360°.  32 entries — one per real tooth.
# Pattern: 11 teeth | 20° gap | 11 teeth | 20° gap | 10 teeth | 30° sync gap
TOOTH_ANGLES_DEG = np.array([
    13.5, 10., 10., 10., 10., 10., 10., 10., 10., 10.,    # teeth  0-9
    16.5,                                                 # tooth 10  (2-slot gap -> 20°)
    13.5, 10., 10., 10., 10., 10., 10., 10., 10., 10.,    # teeth 11-20
    16.5,                                                 # tooth 21  (2-slot gap -> 20°)
    13.5, 10., 10., 10., 10., 10., 10., 10., 10.,         # teeth 22-30
    26.5,                                                 # tooth 31  (3-slot sync gap -> 30°)
])
assert len(TOOTH_ANGLES_DEG) == 32, f"Expected 32 angles, got {len(TOOTH_ANGLES_DEG)}"
assert abs(TOOTH_ANGLES_DEG.sum() - 360.0) < 1e-6, f"Angles sum to {TOOTH_ANGLES_DEG.sum():.4f}°, expected 360°"
print("Tooth angles [deg]:", TOOTH_ANGLES_DEG)
print(f"Sum = {TOOTH_ANGLES_DEG.sum():.1f}°")


# ── Wheel descriptor ───────────────────────────────────────────────────────────
WheelDescriptor = namedtuple('WheelDescriptor', [
    'tooth_angles_deg',  # np.ndarray: crank angle [deg] from each tooth to the next
])

WHEEL_6G75 = WheelDescriptor(
    tooth_angles_deg = TOOTH_ANGLES_DEG,
)

# Choose an RPM profile - uncomment the one you want:
# syn_rpm_profile = rpm_const(SYN_RPM)
# syn_rpm_profile = rpm_linear(SYN_RPM, accel_rpm_per_rev=20)          # constant acceleration: +20 RPM/rev
# syn_rpm_profile = rpm_sine(SYN_RPM, amplitude=90, period_deg=120, phase_deg=120)
RAMP_UP_MS = 300.0
RAMP_UP_ACCEL_RPM_PER_MS = 20.0
RAMP_HOLD_MS = 700.0
RAMP_DOWN_MS = 300.0
RAMP_DOWN_ACCEL_RPM_PER_MS = -30.0

ramp_up_end_rpm = SYN_RPM + RAMP_UP_ACCEL_RPM_PER_MS * RAMP_UP_MS
ramp_down_end_rpm = max(ramp_up_end_rpm + RAMP_DOWN_ACCEL_RPM_PER_MS * RAMP_DOWN_MS, 1.0)

syn_rpm_profile = rpm_staged_ms([
    (RAMP_UP_MS, rpm_linear_ms(SYN_RPM, accel_rpm_per_ms=RAMP_UP_ACCEL_RPM_PER_MS)),
    (RAMP_HOLD_MS, rpm_const(ramp_up_end_rpm)),
    (RAMP_DOWN_MS, rpm_linear_ms(ramp_up_end_rpm, accel_rpm_per_ms=RAMP_DOWN_ACCEL_RPM_PER_MS)),
    (None, rpm_const(ramp_down_end_rpm)),
])
# syn_rpm_profile = rpm_sine(SYN_RPM, amplitude=30, period_deg=180)    # +-30 RPM sine, 2 cycles/rev

# ── Generate waveform & run filter ────────────────────────────────────────────
syn_times, syn_intervals = gen_synthetic_waveform(syn_rpm_profile, SYN_N_REVS, WHEEL_6G75)
syn_signal_times, syn_signal_values = gen_synthetic_signal(syn_times, syn_intervals)

_n = len(WHEEL_6G75.tooth_angles_deg)
_angles = WHEEL_6G75.tooth_angles_deg[np.arange(len(syn_intervals)) % _n]
rpm_from_intervals = 60_000.0 / (360.0 * syn_intervals / _angles)
print(f"Synthetic signal: {SYN_N_REVS} revolutions, {len(syn_times)} edges,  duration = {syn_times.iloc[-1]:.1f} ms")
print(f"RPM range: {rpm_from_intervals.min():.1f} - {rpm_from_intervals.max():.1f}")
print(f"Ramp-up end RPM: {ramp_up_end_rpm:.1f}; Ramp-down end RPM: {ramp_down_end_rpm:.1f}")


Tooth angles [deg]: [13.5 10.  10.  10.  10.  10.  10.  10.  10.  10.  16.5 13.5 10.  10.
 10.  10.  10.  10.  10.  10.  10.  16.5 13.5 10.  10.  10.  10.  10.
 10.  10.  10.  26.5]
Sum = 360.0°
Synthetic signal: 100 revolutions, 3200 edges,  duration = 1144.7 ms
RPM range: 300.0 - 6300.0
Ramp-up end RPM: 6300.0; Ramp-down end RPM: 1.0


In [68]:
files = [
	'6g75-without-spark-crank.csv',
	'6g75-withsparkplugs-cranking.csv',
	'3000gt_crank_cam_cranking_2.csv',
]

csv_path = r"../../unit_tests/tests/trigger/resources/" + files[2]
df = pd.read_csv(csv_path)

df.head()

n_from = 2
n_to = 2400

time = df.iloc[n_from:n_to, 0]
ch0 = df.iloc[n_from:n_to, 1]

signal = df.iloc[n_from:n_to, 1].reset_index(drop=True)
time_arr = df.iloc[n_from:n_to, 0].reset_index(drop=True)

# or use synthetic signal:
signal = syn_signal_values
time_arr = syn_signal_times


time_ms = time_arr * 1000  # convert to milliseconds

In [69]:

fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.08, subplot_titles=("Channel 0", "Channel 1"))

fig.add_trace(go.Scatter(x=time_ms, y=signal, mode='lines', line=dict(width=1, shape='hv'), name='Channel 0'), row=1, col=1)

fig.update_layout(title="6G75 Trigger Signals", height=500, hovermode='x unified')
fig.update_xaxes(title_text="Time [ms]", row=1, col=1)

fig.show()


In [70]:
# Fast rising-edge detection and bounded plotting
EDGE_THRESHOLD = 0.5
MAX_PLOT_EDGES = 100  # Set to None to plot all detected edges

# Use NumPy arrays for faster edge detection than pandas diff/masking
signal_arr = np.asarray(signal, dtype=float)
time_ms_arr = np.asarray(time_ms, dtype=float)

rising_idx = np.flatnonzero((signal_arr[1:] > EDGE_THRESHOLD) & (signal_arr[:-1] <= EDGE_THRESHOLD)) + 1
rising_times_arr = time_ms_arr[rising_idx]
rising_times_ms = pd.Series(rising_times_arr)

total_edges = len(rising_times_arr)
if MAX_PLOT_EDGES is None or MAX_PLOT_EDGES <= 0 or MAX_PLOT_EDGES >= total_edges:
    plot_edges = total_edges
else:
    plot_edges = int(MAX_PLOT_EDGES)

print(f"Found {total_edges} rising edges")
if plot_edges < total_edges:
    print(f"Plotting first {plot_edges} rising edges (set MAX_PLOT_EDGES=None to plot all)")

plot_rising_idx = rising_idx[:plot_edges]
plot_rising_times = time_ms_arr[plot_rising_idx]

# Limit the actual plotted waveform data to the same edge window (+1 edge interval margin).
if plot_edges > 0:
    first_plot_idx = int(plot_rising_idx[0])
    if plot_edges < total_edges:
        next_after_last = int(rising_idx[plot_edges])
        last_plot_idx = min(len(signal_arr), next_after_last + 1)
    else:
        last_plot_idx = len(signal_arr)
else:
    first_plot_idx = 0
    last_plot_idx = len(signal_arr)

plot_time_arr = time_ms_arr[first_plot_idx:last_plot_idx]
plot_signal_arr = signal_arr[first_plot_idx:last_plot_idx]

fig2 = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("Signal with rising edges", "Period between rising edges", "Period ratio (current / previous)", "Ratio of ratios (2nd derivative)")
)

fig2.add_trace(
    go.Scatter(
        x=plot_time_arr, y=plot_signal_arr,
        mode='lines', line=dict(width=1, shape='hv'), name='Channel 0'
    ),
    row=1, col=1
)

# Use one vline shape per displayed edge only (major plotting cost when unbounded)
for t in plot_rising_times:
    fig2.add_shape(
        type='line', x0=t, x1=t, y0=0, y1=1, xref='x', yref='paper',
        opacity=0.3, line=dict(color='red', width=1, dash='dash')
    )

fig2.add_trace(
    go.Scatter(
        x=plot_rising_times,
        y=np.full(plot_edges, float(np.max(plot_signal_arr))) if plot_edges > 0 else [],
        mode='markers', marker=dict(color='red', symbol='triangle-down', size=10),
        name=f'Rising edges shown ({plot_edges}/{total_edges})'
    ),
    row=1, col=1
)

# Compute period features from displayed edges to keep plotting bounded
periods_ms = pd.Series(np.diff(plot_rising_times))
period_times_ms = pd.Series(plot_rising_times[1:])
ratios = periods_ms / periods_ms.shift(1)
ratio_of_ratios = ratios / ratios.shift(1)

fig2.add_trace(go.Scatter(
    x=period_times_ms, y=periods_ms,
    mode='lines+markers', marker=dict(size=4), line=dict(width=1, shape='hv'),
    name='Period [ms]'
), row=2, col=1)

fig2.add_trace(go.Scatter(
    x=period_times_ms, y=ratios,
    mode='lines+markers', marker=dict(size=4), line=dict(width=1, shape='hv'),
    name='Period ratio'
), row=3, col=1)

fig2.add_trace(go.Scatter(
    x=period_times_ms, y=ratio_of_ratios,
    mode='lines+markers', marker=dict(size=4), line=dict(width=1, shape='hv'),
    name='Ratio of ratios'
), row=4, col=1)

fig2.update_yaxes(title_text="Signal", row=1, col=1)
fig2.update_yaxes(title_text="Period [ms]", row=2, col=1)
fig2.update_yaxes(title_text="Ratio", row=3, col=1)
fig2.update_yaxes(title_text="Ratio^2", row=4, col=1)
fig2.update_xaxes(title_text="Time [ms]", row=4, col=1)
fig2.update_layout(title="6G75 Trigger Signal - Rising Edges, Periods & Ratios", height=1000, hovermode='x unified')
fig2.show()


Found 3200 rising edges
Plotting first 100 rising edges (set MAX_PLOT_EDGES=None to plot all)


In [71]:
# ── Alpha-beta filter ──────────────────────────────────────────────────────────
def run_alpha_beta(edge_times_ms, wheel, tooth_offset=0, alpha=0.3, beta=0.05, soft_gate_frac=0.5):
    """
    Run an alpha-beta filter over detected rising-edge times.

    The filter tracks the wheel's angular velocity expressed as 'r': the time
    per degree [ms/deg].  From r you can derive RPM:
        RPM = 60_000 / (360 * r)

    The "hat" (^) notation comes from estimation theory: x̂ denotes an *estimate*
    of the true quantity x, as opposed to the raw noisy measurement.

    Parameters
    ----------
    edge_times_ms  : array of rising-edge timestamps [ms]
    wheel          : WheelDescriptor — geometry of the trigger wheel
    tooth_offset   : index of the first tooth in wheel.tooth_angles_deg to assign to edge 0
    alpha          : position (r̂) correction gain  [0–1]
                     Higher → faster response, more noise
    beta           : velocity (ṙ̂) correction gain  [0–1]
                     Higher → tracks acceleration faster, less stable
    soft_gate_frac : soft-gate width as a fraction of the current r̂
                     Controls how aggressively large innovations are clipped

    Returns
    -------
    dict with keys: edge_times, innov, r_hat, r_dot, tooth_idx, rpm, rev_times
    """
    times   = np.asarray(edge_times_ms)
    n_teeth = len(wheel.tooth_angles_deg)
    tooth_idx = tooth_offset % n_teeth

    # r̂  — estimated time per degree [ms/deg].
    r_hat     = 0.0

    # ṙ̂  — estimated *rate of change* of r̂ [ms/deg per tooth].
    r_dot_hat = 0.0

    innovations, r_hat_log, r_dot_log, tooth_idx_log = [], [], [], []

    r_arr = []

    for k in range(1, len(times)):
        dt = times[k] - times[k - 1]   # measured inter-edge interval [ms]

        # angle_deg — crank angle spanned by this tooth [degrees].
        angle_deg = wheel.tooth_angles_deg[tooth_idx]

        # innov (innovation) = measured r  −  predicted r̂  [ms/deg]
        # A positive innovation means the wheel is slower than predicted;
        # negative means faster.  This is the new information each edge brings.
        innov = dt / angle_deg - r_hat

        # Soft gate: compress large innovations with tanh to reject outliers.
        # sigma sets the half-width of the linear region.
        sigma = soft_gate_frac * r_hat

        if sigma < 1e-9:
            gated_innov = innov
        else:
            gated_innov = sigma * np.tanh(innov / sigma)

        # Alpha-beta update:
        #   r̂  ← r̂ + α · innovation
        #   ṙ̂  ← ṙ̂ + β · innovation
        r_hat     += alpha * gated_innov
        r_dot_hat += beta  * gated_innov

        innovations.append(innov)
        r_hat_log.append(r_hat)
        r_dot_log.append(r_dot_hat)
        tooth_idx_log.append(tooth_idx)

        r_arr.append(dt / angle_deg)

        # Predict forward by one tooth.
        r_hat    += r_dot_hat
        tooth_idx = (tooth_idx + 1) % n_teeth

    innov_arr     = np.array(innovations)
    r_hat_arr     = np.array(r_hat_log)
    r_dot_arr     = np.array(r_dot_log)
    tooth_idx_arr = np.array(tooth_idx_log)
    out_times     = times[1:]
    rev_times     = out_times[tooth_idx_arr == n_teeth - 1]
    rpm_arr       = 60_000.0 / (360.0 * r_hat_arr)

    return dict(
        edge_times=out_times,
        innov=innov_arr,
        r_hat=r_hat_arr,
        r_dot=r_dot_arr,
        tooth_idx=tooth_idx_arr,
        rpm=rpm_arr,
        rev_times=rev_times,
        r_arr=np.array(r_arr)
    )


# ── Plot ───────────────────────────────────────────────────────────────────────
def plot_alpha_beta(result, signal_time_ms, signal, title="Alpha-Beta Tracking"):
    """
    Plot signal, innovation, RPM, and acceleration from run_alpha_beta output.
    Vertical markers are drawn at each wheel revolution (tooth 0).
    """
    edge_times = result['edge_times']
    rev_times  = result['rev_times']

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.07,
        subplot_titles=(
            "Signal",
            "Innovation  (dt/angle_deg − r̂)  [ms/deg]",
            "Angular velocity  [RPM]",
            "Angular acceleration  ṙ  [ms/deg²]"
        )
    )

    fig.add_trace(go.Scatter(x=signal_time_ms, y=signal, mode='lines', line=dict(width=1, shape='hv'), name='Signal'), row=1, col=1)

    fig.add_trace(go.Scatter(x=edge_times, y=result['innov'], mode='lines+markers', marker=dict(size=3), line=dict(width=1, shape='hv'), name='Innovation'), row=2, col=1)
    fig.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=2, col=1)

    fig.add_trace(go.Scatter(x=edge_times, y=result['rpm'], mode='lines+markers', marker=dict(size=3), line=dict(width=1, shape='hv'), name='RPM'), row=3, col=1)

    fig.add_trace(go.Scatter(x=edge_times, y=result['r_dot'], mode='lines+markers', marker=dict(size=3), line=dict(width=1, shape='hv'), name='ṙ (accel)'), row=4, col=1)
    fig.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=4, col=1)

    for t in rev_times:
        fig.add_shape(
            type='line', x0=t, x1=t, y0=0, y1=1,
            xref='x', yref='paper',
            opacity=0.4, line=dict(color='red', width=1.5, dash='dash')
        )

    fig.update_yaxes(title_text="Signal",          row=1, col=1)
    fig.update_yaxes(title_text="Innov [ms/deg]",  row=2, col=1)
    fig.update_yaxes(title_text="RPM",             row=3, col=1)
    fig.update_yaxes(title_text="ṙ [ms/deg²]",    row=4, col=1)
    fig.update_xaxes(title_text="Time [ms]",        row=4, col=1)
    fig.update_layout(title=title, height=1000, hovermode='x unified')
    fig.show()

print("Defined: WheelDescriptor, WHEEL_6G75")
print("Functions defined: run_alpha_beta(), plot_alpha_beta()")


Defined: WheelDescriptor, WHEEL_6G75
Functions defined: run_alpha_beta(), plot_alpha_beta()


In [72]:

# ── User-configurable parameters ───────────────────────────────────────────────
ALPHA          = 0.6   # period (r̂) gain
BETA           = 0.05  # acceleration (ṙ̂) gain
SOFT_GATE_FRAC = 0.6   # soft-gate threshold as fraction of r̂

# ── Find tooth offset with minimum RMS innovation ─────────────────────────────
_n_teeth = len(WHEEL_6G75.tooth_angles_deg)
rms_per_offset = []
for i in range(_n_teeth):
    res = run_alpha_beta(
        rising_times_ms,
        wheel=WHEEL_6G75,
        tooth_offset=i,
        alpha=ALPHA,
        beta=BETA,
        soft_gate_frac=SOFT_GATE_FRAC,
    )
    rms_per_offset.append(np.sqrt(np.mean(res['innov'] ** 2)))

rms_arr = np.array(rms_per_offset)
best_offset = int(np.argmin(rms_arr))
print(f"Best tooth offset: {best_offset}  (RMS innovation = {rms_arr[best_offset]:.4f} ms/deg)")
print(f"Worst tooth offset: {int(np.argmax(rms_arr))}  (RMS innovation = {rms_arr.max():.4f} ms/deg)")

# ── Plot RMS vs tooth offset ──────────────────────────────────────────────────
fig_rms = go.Figure()
fig_rms.add_trace(go.Bar(x=list(range(_n_teeth)), y=rms_arr, name='RMS innovation'))
fig_rms.add_vline(x=best_offset, line=dict(color='red', dash='dash'), annotation_text=f'best={best_offset}')
fig_rms.update_layout(
    title="RMS of innovation vs tooth offset",
    xaxis_title="Tooth offset",
    yaxis_title="RMS innovation [ms/deg]",
    height=350,
)
fig_rms.show()

# ── Plot the best offset ──────────────────────────────────────────────────────
best_result = run_alpha_beta(
    rising_times_ms,
    wheel=WHEEL_6G75,
    tooth_offset=best_offset,
    alpha=ALPHA,
    beta=BETA,
    soft_gate_frac=SOFT_GATE_FRAC,
)
print(f"Detected {len(best_result['rev_times'])} wheel revolutions")
plot_alpha_beta(
    best_result, time_ms, signal,
    title=f"6G75 Alpha-Beta Tracking — best offset={best_offset}  (α={ALPHA}, β={BETA})"
)


Best tooth offset: 0  (RMS innovation = 10.3934 ms/deg)
Worst tooth offset: 30  (RMS innovation = 20.9059 ms/deg)


Detected 99 wheel revolutions


In [73]:
# ── Synthetic grid search: RMSE vs ground truth ───────────────────────────────
# Since we generated syn_times/syn_intervals, we know the true r at every tooth:
#   r_true[i] = actual interval / angle_deg = true ms-per-deg at tooth i
#
# This is an unbiased metric: it doesn't depend on r_hat at all, so a filter
# that tracks better (including with useful beta) will score lower here.

RUN_SYNTHETIC_GRID_SEARCH = False


def run_synthetic_grid_search(
    syn_times,
    syn_intervals,
    wheel,
    soft_gate_frac,
    syn_n_revs,
    syn_signal_times,
    syn_signal_values,
    syn_tooth_offset_gs=0,
    alpha_values=np.linspace(0.0001, 1.5, 50),
    beta_values=np.linspace(0.0001, 1.5, 51),
):
    _n = len(wheel.tooth_angles_deg)
    r_true = np.array([
        syn_intervals.iloc[i] / wheel.tooth_angles_deg[
            (syn_tooth_offset_gs + i) % _n
        ]
        for i in range(len(syn_times) - 1)
    ])

    rmse_grid_syn = np.zeros((len(beta_values), len(alpha_values)))

    for bi, beta in enumerate(beta_values):
        for ai, alpha in enumerate(alpha_values):
            res = run_alpha_beta(
                syn_times,
                wheel=wheel,
                tooth_offset=syn_tooth_offset_gs,
                alpha=alpha,
                beta=beta,
                soft_gate_frac=soft_gate_frac,
            )
            rmse_grid_syn[bi, ai] = np.sqrt(np.mean((res['r_hat'] - r_true) ** 2))

    best_idx_syn = np.unravel_index(np.argmin(rmse_grid_syn), rmse_grid_syn.shape)
    best_beta_syn = beta_values[best_idx_syn[0]]
    best_alpha_syn = alpha_values[best_idx_syn[1]]
    best_rmse_syn = rmse_grid_syn[best_idx_syn]
    print(f"[Synthetic RMSE vs ground truth]  best α={best_alpha_syn:.4f}, β={best_beta_syn:.4f},  RMSE={best_rmse_syn:.6f} ms/deg")

    fig_syn_gs = go.Figure(go.Heatmap(
        z=rmse_grid_syn,
        x=np.round(alpha_values, 4),
        y=np.round(beta_values, 4),
        colorscale='Viridis_r',
        colorbar=dict(title='RMSE<br>[ms/deg]'),
    ))
    fig_syn_gs.add_trace(go.Scatter(
        x=[best_alpha_syn], y=[best_beta_syn],
        mode='markers',
        marker=dict(color='red', size=12, symbol='x'),
        name=f'best (α={best_alpha_syn:.3f}, β={best_beta_syn:.3f})',
    ))
    fig_syn_gs.update_layout(
        title=f"Synthetic: RMSE vs ground-truth r  (RPM profile: {syn_n_revs} revs, sine ±50 RPM @ 120°)",
        xaxis_title="alpha (α)",
        yaxis_title="beta (β)",
        height=500,
    )
    fig_syn_gs.show()

    syn_best_result = run_alpha_beta(
        syn_times,
        wheel=wheel,
        tooth_offset=syn_tooth_offset_gs,
        alpha=best_alpha_syn,
        beta=best_beta_syn,
        soft_gate_frac=soft_gate_frac,
    )
    plot_alpha_beta(
        syn_best_result, syn_signal_times, syn_signal_values,
        title=f"Synthetic - ground-truth best  α={best_alpha_syn:.3f}, β={best_beta_syn:.3f}  (RMSE={best_rmse_syn:.5f})"
    )

    return {
        'r_true': r_true,
        'alpha_values': alpha_values,
        'beta_values': beta_values,
        'rmse_grid_syn': rmse_grid_syn,
        'best_idx_syn': best_idx_syn,
        'best_alpha_syn': best_alpha_syn,
        'best_beta_syn': best_beta_syn,
        'best_rmse_syn': best_rmse_syn,
        'syn_best_result': syn_best_result,
    }


syn_grid_search_result = None
best_alpha_syn = None
best_beta_syn = None
best_rmse_syn = None

if RUN_SYNTHETIC_GRID_SEARCH:
    syn_grid_search_result = run_synthetic_grid_search(
        syn_times=syn_times,
        syn_intervals=syn_intervals,
        wheel=WHEEL_6G75,
        soft_gate_frac=SOFT_GATE_FRAC,
        syn_n_revs=SYN_N_REVS,
        syn_signal_times=syn_signal_times,
        syn_signal_values=syn_signal_values,
        syn_tooth_offset_gs=0,
    )
    best_alpha_syn = syn_grid_search_result['best_alpha_syn']
    best_beta_syn = syn_grid_search_result['best_beta_syn']
    best_rmse_syn = syn_grid_search_result['best_rmse_syn']
else:
    print("Synthetic grid search skipped (set RUN_SYNTHETIC_GRID_SEARCH = True to run).")


Synthetic grid search skipped (set RUN_SYNTHETIC_GRID_SEARCH = True to run).


In [74]:
# ── Grid search: lag-1 autocorrelation of innovations ─────────────────────────
# In an optimally-tuned filter the innovations are white noise (uncorrelated).
#   autocorr > 0  -> filter lags reality (too slow - beta should help here)
#   autocorr < 0  -> filter overshoots (too aggressive)
#   autocorr ≈ 0  -> filter is well-matched to the signal dynamics
#
# This metric works on real data without ground truth.

RUN_AUTOCORR_GRID_SEARCH = False


def run_autocorr_grid_search(
    rising_times_ms,
    wheel,
    best_offset,
    soft_gate_frac,
    time_ms,
    signal,
    alpha_values=np.linspace(0.05, 1, 100),
    beta_values=np.linspace(0.0, 1, 101),
):
    ac_grid = np.zeros((len(beta_values), len(alpha_values)))

    for bi, beta in enumerate(beta_values):
        for ai, alpha in enumerate(alpha_values):
            res = run_alpha_beta(
                rising_times_ms,
                wheel=wheel,
                tooth_offset=best_offset,
                alpha=alpha,
                beta=beta,
                soft_gate_frac=soft_gate_frac,
            )
            innov = res['innov']
            ac_grid[bi, ai] = np.corrcoef(innov[:-1], innov[1:])[0, 1] if len(innov) > 2 else 0.0

    best_idx_ac = np.unravel_index(np.argmin(np.abs(ac_grid)), ac_grid.shape)
    best_beta_ac = beta_values[best_idx_ac[0]]
    best_alpha_ac = alpha_values[best_idx_ac[1]]
    print(f"[Autocorr -> 0]  best α={best_alpha_ac:.4f}, β={best_beta_ac:.4f},  lag-1 autocorr={ac_grid[best_idx_ac]:.4f}")

    fig_ac = go.Figure(go.Heatmap(
        z=ac_grid,
        x=np.round(alpha_values, 4),
        y=np.round(beta_values, 4),
        colorscale='RdBu',
        zmid=0,
        colorbar=dict(title='Lag-1<br>autocorr'),
    ))
    fig_ac.add_trace(go.Scatter(
        x=[best_alpha_ac], y=[best_beta_ac],
        mode='markers',
        marker=dict(color='lime', size=12, symbol='x'),
        name=f'|autocorr|->0  (α={best_alpha_ac:.3f}, β={best_beta_ac:.3f})',
    ))
    fig_ac.update_layout(
        title="Lag-1 autocorrelation of innovations vs α & β  (target: 0 = white innovations)",
        xaxis_title="alpha (α)",
        yaxis_title="beta (β)",
        height=500,
    )
    fig_ac.show()

    ac_result = run_alpha_beta(
        rising_times_ms,
        wheel=wheel,
        tooth_offset=best_offset,
        alpha=best_alpha_ac,
        beta=best_beta_ac,
        soft_gate_frac=soft_gate_frac,
    )
    plot_alpha_beta(
        ac_result, time_ms, signal,
        title=f"Best by autocorr whiteness - α={best_alpha_ac:.3f}, β={best_beta_ac:.3f}  (autocorr={ac_grid[best_idx_ac]:.4f})"
    )

    return {
        'alpha_values': alpha_values,
        'beta_values': beta_values,
        'ac_grid': ac_grid,
        'best_idx_ac': best_idx_ac,
        'best_alpha_ac': best_alpha_ac,
        'best_beta_ac': best_beta_ac,
        'ac_result': ac_result,
    }


autocorr_grid_search_result = None
best_alpha_ac = None
best_beta_ac = None

if RUN_AUTOCORR_GRID_SEARCH:
    autocorr_grid_search_result = run_autocorr_grid_search(
        rising_times_ms=rising_times_ms,
        wheel=WHEEL_6G75,
        best_offset=best_offset,
        soft_gate_frac=SOFT_GATE_FRAC,
        time_ms=time_ms,
        signal=signal,
    )
    best_alpha_ac = autocorr_grid_search_result['best_alpha_ac']
    best_beta_ac = autocorr_grid_search_result['best_beta_ac']
else:
    print("Autocorr grid search skipped (set RUN_AUTOCORR_GRID_SEARCH = True to run).")


Autocorr grid search skipped (set RUN_AUTOCORR_GRID_SEARCH = True to run).


In [75]:
# ── rusEFI-style trigger decoder prototype — 6G75 ────────────────────────────
#
# Mirrors the MCU inner loop (trigger_decoder.cpp) as closely as possible:
#   * Stream processing: on_edge(dt_ms) called once per rising edge
#   * No numpy in the hot path — pure scalar arithmetic
#   * Pre-sync:  raw ratio-window check  (existing isSyncPoint logic)
#   * Post-sync: alpha-beta tracker with per-tooth angle model
#                (design-doc Steps 1-3; tracker prediction replaces
#                 toothDurations[1] as the ratio denominator)
#
# State variables map directly to TriggerDecoderBase fields.
# float ms throughout  ≡  uint32 timer ticks on MCU.

def make_decoder_6g75(wheel,
                      sync_ratio_from = 2.3,   # ratio window for 3-slot sync gap
                      sync_ratio_to   = 3.6,
                      alpha_rate      = 0.125,  # rate EMA gain   (1/8  → right-shift 3)
                      alpha_trend     = 0.0625, # trend EMA gain  (1/16 → right-shift 4)
                      alpha_var       = 0.0625, # variance EMA gain
                      gate_k          = 9,      # glitch gate multiplier (≈ 3σ²)
                      var_floor_frac  = 0.02,   # sync_var floor = (var_floor_frac * t_hat)²
                                                # prevents gate from narrowing below wheel tolerance
                      tracker_warmup  = 4):     # teeth before gate is armed
    """
    Returns a callable  on_edge(dt_ms) -> dict
    that processes one inter-edge interval at a time.

    State is held in closure variables, mirroring TriggerDecoderBase:
        dt_cur / dt_prev  — toothDurations[0] / [1]
        t_hat             — EMA of rate [ms/deg]; holds PREDICTED value between calls
        t_dot_hat         — EMA of rate trend [ms/deg per tooth]
        sync_var          — EMA of innovation², clamped ≥ (var_floor_frac·t_hat)²
        tooth_idx         — index into wheel.tooth_angles_deg: angle FROM this tooth to next

    var_floor_frac: the gate width can never shrink below var_floor_frac·r̂.
        At 2% (default), a 2% relative timing error is always considered normal
        and will never trip the glitch gate regardless of how smooth the run is.
        This is important for real wheels whose wide/missing-tooth intervals
        differ from the calibrated angles by more than the sensor jitter on
        regular teeth.
    """
    # The sync gap is the last tooth's angle (tooth 31: 30°)
    sync_gap_deg = float(wheel.tooth_angles_deg[-1])
    n_teeth = len(wheel.tooth_angles_deg)

    # ── mutable state ─────────────────────────────────────────────────────────
    dt_cur      = 0.0
    dt_prev     = 0.0
    synced      = False
    tooth_idx   = 0
    t_hat       = 0.0
    t_dot_hat   = 0.0
    sync_var    = 0.0
    tracker_ok  = False
    tracker_n   = 0

    def on_edge(dt):
        nonlocal dt_cur, dt_prev, synced, tooth_idx
        nonlocal t_hat, t_dot_hat, sync_var, tracker_ok, tracker_n

        # Shift duration buffer  (toothDurations[] shift in MCU)
        dt_prev = dt_cur
        dt_cur  = dt

        innov     = 0.0
        is_sync   = False
        is_glitch = False

        if not synced:
            # ── PRE-SYNC: ratio-window check (isSyncPoint equivalent) ─────
            if dt_prev > 0.0:
                ratio = dt_cur / dt_prev
                if sync_ratio_from <= ratio <= sync_ratio_to:
                    synced     = True
                    tooth_idx  = 0
                    is_sync    = True
                    # Bootstrap: sync gap spans sync_gap_deg degrees,
                    # so t_hat = observed ms/deg for that gap.
                    # This becomes the PREDICTED rate for the first post-sync tooth.
                    t_hat      = dt / sync_gap_deg
                    t_dot_hat  = 0.0
                    sync_var   = 0.0
                    tracker_n  = 0
                    tracker_ok = False

            r_hat_out = t_hat

        else:
            # ── POST-SYNC: Steps 1-3 from design doc ─────────────────────

            # Step 1 — Predict
            # t_hat already holds the PREDICTED value advanced by t_dot_hat
            # at the end of the previous call (mirrors r_hat += r_dot_hat in MCU).
            angle_deg = wheel.tooth_angles_deg[tooth_idx]
            r_pred = t_hat
            r_meas = dt / angle_deg    # measured ms/deg
            innov  = r_meas - r_pred   # innovation [ms/deg]

            # Step 2 — Glitch gate
            is_glitch = tracker_ok and (innov * innov) > gate_k * sync_var

            # Step 3 — Update (always update rate/var; only count non-glitches for warmup)
            t_hat     += alpha_rate  * innov
            t_dot_hat += alpha_trend * innov
            sync_var  += alpha_var * (innov * innov - sync_var)

            # Floor: sync_var ≥ (var_floor_frac · t_hat)²
            # This ensures the gate band is always at least var_floor_frac·r̂ wide,
            # regardless of how many smooth teeth were seen in a row.
            floor = (var_floor_frac * t_hat) ** 2
            if sync_var < floor:
                sync_var = floor

            if not is_glitch:
                tracker_n += 1
                if tracker_n >= tracker_warmup:
                    tracker_ok = True

            r_hat_out = t_hat      # log AFTER update, BEFORE predict (matches run_alpha_beta)

            # Predict-step: advance t_hat by trend for next tooth
            t_hat    += t_dot_hat
            tooth_idx = (tooth_idx + 1) % n_teeth

        rpm = 60_000.0 / (360.0 * t_hat) if t_hat > 0.0 else 0.0
        return {
            'synced':    synced,
            'is_sync':   is_sync,
            'is_glitch': is_glitch,
            'tooth_idx': tooth_idx,
            'innov':     innov,
            'r_hat':     r_hat_out,
            't_dot':     t_dot_hat,
            'sync_var':  sync_var,
            'rpm':       rpm,
        }

    return on_edge


print("Defined: make_decoder_6g75()")
print("Usage:  decoder = make_decoder_6g75(WHEEL_6G75)")
print("        entry   = decoder(dt_ms)   # once per rising edge")


Defined: make_decoder_6g75()
Usage:  decoder = make_decoder_6g75(WHEEL_6G75)
        entry   = decoder(dt_ms)   # once per rising edge


In [ ]:

# ── Run prototype decoder and plot ────────────────────────────────────────────

PROTO_ALPHA_RATE    = 0.125
PROTO_ALPHA_TREND   = 0.125
PROTO_ALPHA_VAR     = 0.0625
PROTO_GATE_K        = 9
PROTO_VAR_FLOOR_FRAC = 0.02   # gate never narrows below 2% of r̂  (≈ wheel tolerance floor)

decoder = make_decoder_6g75(
    WHEEL_6G75,
    alpha_rate     = PROTO_ALPHA_RATE,
    alpha_trend    = PROTO_ALPHA_TREND,
    alpha_var      = PROTO_ALPHA_VAR,
    gate_k         = PROTO_GATE_K,
    var_floor_frac = PROTO_VAR_FLOOR_FRAC,
)

# Stream: one inter-edge interval at a time, no bulk numpy operations
times_arr = rising_times_ms.values
log = []
for k in range(1, len(times_arr)):
    dt    = times_arr[k] - times_arr[k - 1]   # one scalar subtract — mirrors MCU
    entry = decoder(dt)
    entry['t'] = times_arr[k]
    log.append(entry)

# ── Collect output into plain lists for plotting ───────────────────────────────
t_out        = [e['t']         for e in log]
rpm_out      = [e['rpm']       for e in log]
innov_out    = [e['innov']     for e in log]
tdot_out     = [e['t_dot']     for e in log]
synced_out   = [e['synced']    for e in log]
glitch_out   = [e['is_glitch'] for e in log]
sync_var_out = [e['sync_var']  for e in log]

import math
gate_thresh = [math.sqrt(PROTO_GATE_K * sv) for sv in sync_var_out]

# First sync-point time
sync_t = next((e['t'] for e in log if e['is_sync']), None)
print(f"Sync detected at t = {sync_t:.3f} ms" if sync_t else "No sync detected")
print(f"Edges after sync: {sum(synced_out)}")
print(f"Glitches flagged: {sum(glitch_out)}")

# ── Reference: run_alpha_beta with matching gains (no soft gate) ───────────────
ref = run_alpha_beta(
    rising_times_ms,
    wheel        = WHEEL_6G75,
    tooth_offset = best_offset,
    alpha        = PROTO_ALPHA_RATE,
    beta         = PROTO_ALPHA_TREND,
    soft_gate_frac = 0.0,   # disable soft gate so comparison is apples-to-apples
)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig_proto = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    subplot_titles=(
        "Signal",
        "Innovation [ms/deg]   — prototype (blue) vs run_alpha_beta (grey)",
        "RPM",
        "Rate trend  ṙ  [ms/deg per tooth]",
    )
)

# Row 1: signal
fig_proto.add_trace(
    go.Scatter(x=time_ms, y=signal, mode='lines',
               line=dict(width=1, shape='hv'), name='Signal'),
    row=1, col=1
)

# Row 2: innovations
# Gate threshold band: ±sqrt(gate_k * sync_var) as a filled interval
fig_proto.add_trace(
    go.Scatter(x=t_out, y=gate_thresh, mode='lines',
               line=dict(width=0), showlegend=False, hoverinfo='skip'),
    row=2, col=1
)
fig_proto.add_trace(
    go.Scatter(x=t_out, y=[-v for v in gate_thresh], mode='lines',
               line=dict(width=0), fill='tonexty',
               fillcolor='rgba(255,165,0,0.25)',
               name='±√(gate_k·var)'),
    row=2, col=1
)

fig_proto.add_trace(
    go.Scatter(x=t_out, y=innov_out, mode='lines+markers',
               marker=dict(size=3), line=dict(width=1, shape='hv'),
               name='Prototype innov'),
    row=2, col=1
)
fig_proto.add_trace(
    go.Scatter(x=ref['edge_times'], y=ref['innov'], mode='lines',
               line=dict(width=1, color='grey', dash='dot'),
               name='run_alpha_beta innov'),
    row=2, col=1
)
fig_proto.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=2, col=1)

# Glitches
glitch_t   = [t_out[i] for i, g in enumerate(glitch_out) if g]
glitch_inn = [innov_out[i] for i, g in enumerate(glitch_out) if g]
if glitch_t:
    fig_proto.add_trace(
        go.Scatter(x=glitch_t, y=glitch_inn, mode='markers',
                   marker=dict(color='red', size=8, symbol='x'), name='Glitch'),
        row=2, col=1
    )

# Row 3: RPM
fig_proto.add_trace(
    go.Scatter(x=t_out, y=rpm_out, mode='lines+markers',
               marker=dict(size=3), line=dict(width=1, shape='hv'),
               name='Prototype RPM'),
    row=3, col=1
)
fig_proto.add_trace(
    go.Scatter(x=ref['edge_times'], y=ref['rpm'], mode='lines',
               line=dict(width=1, color='grey', dash='dot'),
               name='run_alpha_beta RPM'),
    row=3, col=1
)

# Row 4: trend
fig_proto.add_trace(
    go.Scatter(x=t_out, y=tdot_out, mode='lines+markers',
               marker=dict(size=3), line=dict(width=1, shape='hv'),
               name='ṙ (trend)'),
    row=4, col=1
)
fig_proto.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=4, col=1)

# Sync detection marker
if sync_t is not None:
    fig_proto.add_vline(x=sync_t, line=dict(color='green', width=2, dash='dash'),
                        annotation_text='sync', row='all', col='all')

fig_proto.update_yaxes(title_text="Signal",        row=1, col=1)
fig_proto.update_yaxes(title_text="Innov [ms/deg]", row=2, col=1)
fig_proto.update_yaxes(title_text="RPM",           row=3, col=1)
fig_proto.update_yaxes(title_text="ṙ [ms/deg²]",  row=4, col=1)
fig_proto.update_xaxes(title_text="Time [ms]",     row=4, col=1)
fig_proto.update_layout(
    title=f"Prototype decoder — α_rate={PROTO_ALPHA_RATE}, α_trend={PROTO_ALPHA_TREND}, var_floor={PROTO_VAR_FLOOR_FRAC*100:.0f}%",
    height=1000, hovermode='x unified'
)
fig_proto.show()


Sync detected at t = 65020.053 ms
Edges after sync: 3168
Glitches flagged: 0
